# Risk Model Experimentation

Exploration notebook for the risk-scoring ML model used by `backend/app/agents/risk/risk_model.py`. The production training entry point is `pipeline/pipeline.py` — this notebook walks through the same steps interactively so you can inspect the data, try different models, and look at feature importance.

Run this with the backend venv's kernel (`backend/.venv`) so `app.*` imports resolve and scikit-learn/pandas are available.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(REPO_ROOT / "backend"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

from app.agents.risk.feature_engineering import FEATURE_COLUMNS

sys.path.insert(0, str(REPO_ROOT / "pipeline"))
from pipeline import generate_dataset  # reuse the same synthetic-data generator the training script uses

REPO_ROOT

## 1. Generate (or load) the training data

There's no real historical maritime incident dataset yet, so `generate_dataset()` bootstraps one: it samples random event/route/risk combinations across the same categories the agents already reason about, and labels each with the existing rule-based `RiskScorer` plus a little noise. Swap this cell for `pd.read_csv(REPO_ROOT / "data" / "processed" / "risk_features.csv")` once real data exists, or point it at a real incident log.

In [ ]:
df = generate_dataset(n_samples=5000, seed=42)
df.head()

In [ ]:
df.describe()

## 2. Look at feature correlation with the label

Sanity check: severity, likelihood, and impact should dominate, since those drive most of the rule-based label.

In [ ]:
df.corr(numeric_only=True)["risk_score"].sort_values(ascending=False)

## 3. Train/test split and model training

In [ ]:
X = df[FEATURE_COLUMNS]
y = df["risk_score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f"Test MAE: {mean_absolute_error(y_test, predictions):.2f}")
print(f"Test R2:  {r2_score(y_test, predictions):.3f}")

## 4. Feature importance

Which engineered features is the model actually relying on?

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLUMNS).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot.barh(ax=ax)
ax.set_xlabel("Importance")
ax.set_title("Risk model feature importance")
plt.tight_layout()
plt.show()

## 5. Predicted vs. actual

How well does the model track the labels it hasn't seen?

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, predictions, alpha=0.3, s=10)
ax.plot([0, 100], [0, 100], "r--", linewidth=1)
ax.set_xlabel("Actual risk score")
ax.set_ylabel("Predicted risk score")
ax.set_title("Predicted vs. actual")
plt.tight_layout()
plt.show()

## 6. Try it on a single case

Same feature path the live `RiskAgent` uses, so this should match what the API returns for an equivalent event.

In [ ]:
from app.agents.risk.feature_engineering import FeatureEngineer

sample_event = {"severity": "critical", "source": "AIS", "description": "Cyclone in transit corridor"}
sample_route = {"status": "active", "waypoints": ["WP-1", "WP-2", "WP-3"]}

features = FeatureEngineer.combine_features(event=sample_event, route=sample_route)
vector = FeatureEngineer.to_vector(features)
row = pd.DataFrame([vector], columns=FEATURE_COLUMNS)

predicted_score = model.predict(row)[0]
print(f"Predicted risk score: {predicted_score:.1f}")

## 7. Save the model

This is what `pipeline/pipeline.py` does on every run — only execute this cell if you want this notebook's model (e.g. a different algorithm or hyperparameters you tried above) to become the one `RiskAgent` actually loads.

In [ ]:
import joblib

MODEL_PATH = REPO_ROOT / "models" / "saved_models" / "risk_model.joblib"
# joblib.dump(model, MODEL_PATH)
# print(f"Saved -> {MODEL_PATH}")